In [1]:
import joblib

import numpy as np
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer ,PorterStemmer
import spacy

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB  
from hmmlearn.hmm import GaussianHMM

# from sktime.detection.hmm_learn import GaussianHMM 

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.decomposition import TruncatedSVD #instead of PCA
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


In [2]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer", "tagger"])

In [3]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
stop_words=set(stopwords.words('english'))

# Load the data + class Weights

In [5]:
data = joblib.load('news_data.pkl') # original
x_train = data['X_train']
y_train = data['y_train']
x_test = data['X_test']
y_test = data['y_test']

data_balance = joblib.load('news_data_resampled.pkl') # oversample only
x_train_ran_res = data_balance['X_train']
y_train_ran_res = data_balance['y_train']
x_test_ran_res = data_balance['X_test']
y_test_ran_res = data_balance['y_test']

data_balance_rosrus=joblib.load('news_data_bal_ros_rus.pkl')  
x_train_bal = data_balance_rosrus['X_train']
y_train_bal = data_balance_rosrus['y_train']
x_test_bal = data_balance_rosrus['X_test']
y_test_bal = data_balance_rosrus['y_test']

In [6]:
data_undersampled=joblib.load('news_data_undersampled.pkl') # undersample only
x_train_undersampled=data_undersampled['X_train']
y_train_undersampled=data_undersampled['y_train']
x_test_undersampled=data_undersampled['X_test']
y_test_undersampled=data_undersampled['y_test']

In [7]:
class_weights_dict=joblib.load('classWeightsDic')

# Initial  NER

Stopword removal was intentionally not applied to the NER pipeline 
Because NER depends on full contextual information. Stopwords are part of the sentence structure and removing them can break entity boundaries and degrade recognition accuracy.

In [ ]:
def NER_features(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    doc = nlp(text)
    return [ent.label_ for ent in doc.ents]

    # tokens = word_tokenize(text.lower())
    # pos_tags = nltk.pos_tag(tokens)
    # ne_tree = nltk.ne_chunk(pos_tags, binary=False)

    # ner_tokens = []

    # for subtree in ne_tree:
    #     if hasattr(subtree, 'label'):
    #         ner_tokens.append(subtree.label())

    # return ' '.join(ner_tokens)

In [9]:
num_classes = len(set(y_train))  
num_classes

10

# Original Data

In [10]:
len(x_train)

167616

## NER feature Extraction + scalling

## count vectorizer

In [76]:
vectorizer_NER_ = CountVectorizer(tokenizer=lambda x:NER_features(x),lowercase=False)
x_train_NER = vectorizer_NER_.fit_transform(x_train)
x_test_NER= vectorizer_NER_.transform(x_test)

In [77]:
print(x_train_NER)

  (2, 13)	1
  (4, 13)	1
  (5, 13)	1
  (6, 17)	1
  (10, 4)	1
  (10, 11)	1
  (13, 13)	1
  (13, 9)	1
  (14, 11)	1
  (17, 7)	1
  (18, 13)	1
  (18, 9)	1
  (19, 11)	1
  (20, 13)	1
  (22, 13)	1
  (24, 13)	1
  (28, 13)	2
  (29, 13)	1
  (29, 11)	1
  (30, 1)	1
  (31, 11)	1
  (32, 11)	1
  (34, 1)	1
  (35, 13)	1
  (36, 11)	1
  :	:
  (167582, 13)	1
  (167583, 11)	1
  (167585, 11)	2
  (167586, 4)	1
  (167587, 13)	1
  (167591, 13)	1
  (167591, 11)	1
  (167592, 9)	1
  (167593, 13)	1
  (167593, 7)	1
  (167595, 11)	1
  (167597, 13)	1
  (167600, 11)	1
  (167601, 11)	1
  (167602, 13)	2
  (167602, 11)	1
  (167603, 17)	1
  (167608, 11)	1
  (167608, 9)	1
  (167609, 17)	1
  (167610, 11)	2
  (167611, 13)	1
  (167611, 11)	1
  (167613, 11)	1
  (167615, 13)	2


In [78]:
NER_counts = np.asarray(x_train_NER.sum(axis=0)).ravel()
NER_names = vectorizer_NER_.get_feature_names_out()

NER_freq = pd.Series(NER_counts, index=NER_names).sort_values(ascending=False)
NER_freq

PERSON         58060
ORG            48630
GPE            13550
NORP            6503
DATE            5936
WORK_OF_ART     5714
CARDINAL        2983
EVENT           1902
FAC             1008
LOC              981
PRODUCT          510
TIME             484
ORDINAL          452
LAW              373
LANGUAGE          52
MONEY             18
PERCENT           11
QUANTITY           8
dtype: int64

In [79]:
scaler = StandardScaler(with_mean=False)
x_train_NER_scaled = scaler.fit_transform(x_train_NER)
x_test_NER_scaled  = scaler.transform(x_test_NER)

## TF-IDF vectorizer

In [80]:
vectorizer_NER_tfidf = TfidfVectorizer(tokenizer=lambda x: NER_features(x))
x_train_NER_tfidf = vectorizer_NER_tfidf.fit_transform(x_train)
x_test_NER_tfidf = vectorizer_NER_tfidf.transform(x_test)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [81]:
scaler_tfidf = StandardScaler(with_mean=False)
x_train_NER_tfidf_scaled = scaler_tfidf.fit_transform(x_train_NER_tfidf)
x_test_NER_tfidf_scaled  = scaler_tfidf.transform(x_test_NER_tfidf)

## Models

In [82]:
# results_NER_original={}
# joblib.dump(results_NER_original,'results_NER_original')

In [83]:
results_NER_original=joblib.load('results_NER_original')
results_NER_original

{}

In [33]:
x_train_NER_tfidf_scaled.shape

(167616, 18)

In [34]:
x_train_NER_scaled.shape

(167616, 18)

In [84]:
models_NER={
    "SVC_NER": SVC(class_weight=class_weights_dict),
    "MultinomialNB_NER": MultinomialNB(),
    "MLP" :MLPClassifier(hidden_layer_sizes=(36,18),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
    # 'HMM' :GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
    }

## with countvectorizer

In [85]:
for model_name, model in models_NER.items():
    model.fit(x_train_NER_scaled, y_train)

    y_pred_train = model.predict(x_train_NER_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_NER_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name}Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_NER_original[model_name] = accuracy

Results for SVC_NER Training: accuracy=0.1474023959526537
Results for SVC_NERTesting: accuracy=0.14513781171697887
              precision    recall  f1-score   support

           1       0.36      0.21      0.26      7120
           2       0.14      0.62      0.23      3589
           3       0.17      0.44      0.25      3473
           4       0.13      0.17      0.15      1980
           5       0.07      0.04      0.05      1963
           6       0.05      0.03      0.04      1269
           7       0.00      0.00      0.00      1268
           8       0.04      0.28      0.07      1198
           9       0.07      0.05      0.06      1016
          10       0.48      0.00      0.00     19029

    accuracy                           0.15     41905
   macro avg       0.15      0.18      0.11     41905
weighted avg       0.32      0.15      0.10     41905

--------------------------------------------------
Results for MultinomialNB_NER Training: accuracy=0.4249057369224895
Results

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM 

In [86]:
svd_NER = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_NER = svd_NER.fit_transform(x_train_NER)
x_test_svd_NER = svd_NER.transform(x_test_NER)

scaler = StandardScaler()
x_train_svd_NER = scaler.fit_transform(x_train_svd_NER)
x_test_svd_NER = scaler.transform(x_test_svd_NER)

In [87]:
hmm_NER=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [88]:
lengths_train = [1] * x_train_svd_NER.shape[0]
lengths_test = [1] * x_test_svd_NER.shape[0]

In [89]:
hmm_NER.fit(x_train_svd_NER)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [90]:
y_pred_NER_original=hmm_NER.predict(x_train_svd_NER,)
accuracy_hmm_NER_original = accuracy_score(y_train, y_pred_NER_original)   
print(f"training HMM NER Accuracy : {accuracy_hmm_NER_original}")

training HMM NER Accuracy : 0.05639676403207331


In [91]:
y_pred_NER_original=hmm_NER.predict(x_test_svd_NER,lengths=lengths_test)
accuracy_hmm_NER_original = accuracy_score(y_test, y_pred_NER_original)   
print(f"testing HMM NER Accuracy with length : {accuracy_hmm_NER_original}")

testing HMM NER Accuracy with length : 0.04724973153561628


In [92]:
y_pred_NER_original=hmm_NER.predict(x_test_svd_NER,)
accuracy_hmm_NER_original = accuracy_score(y_test, y_pred_NER_original)   
print(f"testing HMM NER Accuracy : {accuracy_hmm_NER_original}")

testing HMM NER Accuracy : 0.05507695979000119


In [103]:
results_NER_original['HMM']=0.05507695979000119

In [93]:
results_NER_original

{'SVC_NER': 0.14513781171697887,
 'MultinomialNB_NER': 0.42605894284691564,
 'MLP': 0.4576542178737621}

## With tf-idf

In [94]:
for model_name, model in models_NER.items():
    model.fit(x_train_NER_tfidf_scaled, y_train)

    y_pred_train = model.predict(x_train_NER_tfidf_scaled)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_NER_tfidf_scaled)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results_NER_original["(tf-idf) "+model_name] = accuracy

Results for SVC_NER Training: accuracy=0.17869415807560138
Results for SVC_NER Testing: accuracy=0.1787853478105238
              precision    recall  f1-score   support

           1       0.37      0.34      0.35      7120
           2       0.14      0.80      0.23      3589
           3       0.20      0.40      0.27      3473
           4       0.12      0.22      0.16      1980
           5       0.10      0.14      0.12      1963
           6       0.04      0.00      0.00      1269
           7       0.06      0.00      0.00      1268
           8       0.04      0.02      0.03      1198
           9       0.05      0.02      0.02      1016
          10       0.64      0.00      0.01     19029

    accuracy                           0.18     41905
   macro avg       0.18      0.19      0.12     41905
weighted avg       0.40      0.18      0.12     41905

--------------------------------------------------
Results for MultinomialNB_NER Training: accuracy=0.3972472794959908
Result

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM

In [95]:
svd_NER_tfidf = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_NER_tfidf = svd_NER_tfidf.fit_transform(x_train_NER_tfidf)
x_test_svd_NER_tfidf = svd_NER_tfidf.transform(x_test_NER_tfidf)

scaler = StandardScaler()
x_train_svd_NER_tfidf_scaled = scaler.fit_transform(x_train_svd_NER_tfidf)
x_test_svd_NER_tfidf_scaled = scaler.transform(x_test_svd_NER_tfidf)

In [96]:
hmm_NER_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

In [97]:
lengths_train = [1] * x_train_svd_NER_tfidf_scaled.shape[0]
lengths_test = [1] * x_test_svd_NER_tfidf_scaled.shape[0]

In [98]:
hmm_NER_tfidf.fit(x_train_svd_NER_tfidf_scaled)

Model is not converging.  Current: 6155999.382109529 is not greater than 6156020.237600187. Delta is -20.855490657500923


GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [99]:
y_pred_NER_original_tfidf=hmm_NER_tfidf.predict(x_train_svd_NER_tfidf_scaled,lengths=lengths_train)
accuracy_hmm_NER_original = accuracy_score(y_train, y_pred_NER_original_tfidf)   
print(f"training HMM NER Accuracy : {accuracy_hmm_NER_original}")

training HMM NER Accuracy : 0.04430961244749904


In [100]:
y_pred_NER_original_tfidf=hmm_NER_tfidf.predict(x_test_svd_NER_tfidf_scaled,lengths=lengths_test)
accuracy_hmm_NER_original = accuracy_score(y_test, y_pred_NER_original_tfidf)   
print(f"testing HMM NER Accuracy : {accuracy_hmm_NER_original}")

testing HMM NER Accuracy : 0.04474406395418208


In [101]:
y_pred_NER_original_tfidf=hmm_NER_tfidf.predict(x_test_svd_NER_tfidf_scaled,)
accuracy_hmm_NER_original = accuracy_score(y_test, y_pred_NER_original_tfidf)   
print(f"testing HMM NER Accuracy : {accuracy_hmm_NER_original}")

testing HMM NER Accuracy : 0.031571411526070875


In [104]:
results_NER_original['(tf-idf) HMM']=0.04474406395418208

## Save results

In [105]:
results_NER_original

{'SVC_NER': 0.14513781171697887,
 'MultinomialNB_NER': 0.42605894284691564,
 'MLP': 0.4576542178737621,
 '(tf-idf) SVC_NER': 0.1787853478105238,
 '(tf-idf) MultinomialNB_NER': 0.39792387543252594,
 '(tf-idf) MLP': 0.46373941057153084,
 'HMM': 0.05507695979000119,
 '(tf-idf) HMM': 0.04474406395418208}

In [106]:
joblib.dump(results_NER_original,'results_NER_original')

['results_NER_original']

In [87]:
results_NER_original=joblib.load('results_NER_original')
results_NER_original

{'SVC_NER StopWords removed': 0.14513781171697887,
 'MultinomialNB_NER StopWords removed': 0.42605894284691564,
 'MLP StopWords removed': 0.4576542178737621,
 'HMM StopWords removed': 0.05507695979000119,
 'SVC_NER StopWords kept': 0.1780694427872569,
 'MultinomialNB_NER StopWords kept': 0.39403412480610905,
 'MLP StopWords kept': 0.4642644075885932,
 'HMM StopWords kept': 0.1699081255220141,
 '(tf-idf) SVC_NER StopWords removed': 0.1787853478105238,
 '(tf-idf) MultinomialNB_NER StopWords removed': 0.39792387543252594,
 '(tf-idf) MLP StopWords removed': 0.46373941057153084,
 '(tf-idf) HMM StopWords removed': 0.04474406395418208,
 '(tf-idf) SVC_NER StopWords kept': 0.1787853478105238,
 '(tf-idf) MultinomialNB_NER StopWords kept': 0.39792387543252594,
 '(tf-idf) MLP StopWords kept': 0.46373941057153084,
 '(tf-idf) HMM StopWords kept': 0.04474406395418208}

In [107]:
with open("results_NER_original.txt", "w", encoding="utf-8") as f:
    for key, value in results_NER_original.items():
        f.write(f"{key}: {value}\n")

# ______________________________________________________________________________________

# resampled Data (undersample)

In [108]:
results_NER_undersample={}
# results_NER_undersample=joblib.load('results_NER_undersample')
# results_NER_undersample

## NER feature Extraction + Scaling

### with countvectorizer

In [109]:
vectorizer_NER_undersample = CountVectorizer(tokenizer=lambda x:NER_features(x))
x_train_NER_undersample = vectorizer_NER_undersample.fit_transform(x_train_undersampled)
x_test_NER_undersample= vectorizer_NER_undersample.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [110]:
NER_counts = np.asarray(x_train_NER_undersample.sum(axis=0)).ravel()
NER_names = vectorizer_NER_undersample.get_feature_names_out()

NER_freq = pd.Series(NER_counts, index=NER_names).sort_values(ascending=False)
NER_freq

PERSON         16988
GPE             7439
ORG             6435
DATE            5441
NORP            3737
CARDINAL        1897
ORDINAL         1084
TIME             594
LOC              399
FAC              149
PRODUCT          116
MONEY             36
EVENT             34
LAW               33
LANGUAGE          28
QUANTITY          25
WORK_OF_ART       16
PERCENT           13
dtype: int64

In [111]:
scaler = StandardScaler(with_mean=False)
x_train_NER_undersample_scaled = scaler.fit_transform(x_train_NER_undersample)
x_test_NER_undersample_scaled  = scaler.transform(x_test_NER_undersample)

In [112]:
x_train_NER_undersample_scaled.shape,x_test_NER_undersample_scaled.shape

((66774, 18), (16694, 18))

### with tf-idf vectorizer

In [113]:
vectorizer_NER_undersample_tfidf = TfidfVectorizer(tokenizer=lambda x:NER_features(x))
x_train_NER_undersample_tfidf = vectorizer_NER_undersample_tfidf.fit_transform(x_train_undersampled)
x_test_NER_undersample_tfidf= vectorizer_NER_undersample_tfidf.transform(x_test_undersampled)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [114]:
scaler_tfidf = StandardScaler(with_mean=False)
x_train_NER_undersample_scaled_tfidf = scaler_tfidf.fit_transform(x_train_NER_undersample_tfidf)
x_test_NER_undersample_scaled_tfidf  = scaler_tfidf.transform(x_test_NER_undersample_tfidf)

In [115]:
x_train_NER_undersample_scaled_tfidf.shape,x_test_NER_undersample_scaled_tfidf.shape

((66774, 18), (16694, 18))

## Models

In [116]:
models_NER_undersample={
    'SVM': SVC(),
    'MultinomialNB': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(32,16),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
}

## with countvectorizer

In [117]:
for model_name, model in models_NER_undersample.items():
    model.fit(x_train_NER_undersample_scaled, y_train_undersampled)

    y_pred_train = model.predict(x_train_NER_undersample_scaled)
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_NER_undersample_scaled)
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    print('-'*50)
    results_NER_undersample['undersample '+model_name] = accuracy

Results for SVM Training: accuracy=0.2329199988019289
Results for SVM Testing: accuracy=0.23415598418593506
              precision    recall  f1-score   support

           1       0.28      0.36      0.32      2000
           2       0.19      0.83      0.31      2000
           3       0.28      0.41      0.33      2000
           4       0.31      0.21      0.25      1980
           5       0.25      0.14      0.18      1963
           6       0.20      0.00      0.00      1269
           7       0.00      0.00      0.00      1268
           8       0.25      0.00      0.00      1198
           9       0.20      0.01      0.02      1016
          10       0.23      0.00      0.01      2000

    accuracy                           0.23     16694
   macro avg       0.22      0.20      0.14     16694
weighted avg       0.23      0.23      0.17     16694

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.15781591637463682
Results for Multi

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Results for MLP Training: accuracy=0.23070356725671667
Results for MLP Testing: accuracy=0.23439559123038217
              precision    recall  f1-score   support

           1       0.28      0.37      0.32      2000
           2       0.19      0.80      0.31      2000
           3       0.28      0.42      0.33      2000
           4       0.28      0.24      0.25      1980
           5       0.25      0.14      0.18      1963
           6       0.00      0.00      0.00      1269
           7       0.00      0.00      0.00      1268
           8       0.40      0.00      0.00      1198
           9       0.38      0.00      0.01      1016
          10       0.27      0.00      0.01      2000

    accuracy                           0.23     16694
   macro avg       0.23      0.20      0.14     16694
weighted avg       0.24      0.23      0.17     16694

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### HMM

In [118]:
hmm_NER_undersample=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,35)

In [119]:
svd_NER = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_NER_undersample = svd_NER.fit_transform(x_train_NER_undersample_scaled)
x_test_svd_NER_undersample = svd_NER.transform(x_test_NER_undersample_scaled)

scaler = StandardScaler()
x_train_svd_NER_undersample = scaler.fit_transform(x_train_svd_NER_undersample)
x_test_svd_NER_undersample = scaler.transform(x_test_svd_NER_undersample)

#### apply HMM

In [120]:
lengths_train = [1] * x_train_svd_NER_undersample.shape[0]
lengths_test = [1] * x_test_svd_NER_undersample.shape[0]

In [121]:
hmm_NER_undersample.fit(x_train_svd_NER_undersample)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [122]:
y_pred_stem_undersample=hmm_NER_undersample.predict(x_train_svd_NER_undersample,lengths=lengths_train)
accuracy_hmm_stem_undersample = accuracy_score(y_train_undersampled, y_pred_stem_undersample)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

training HMM BoW Accuracy with lengths: 0.0


In [123]:
y_pred_stem_undersample=hmm_NER_undersample.predict(x_test_svd_NER_undersample,lengths=lengths_test)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy with lengths: 0.0


In [124]:
y_pred_stem_undersample=hmm_NER_undersample.predict(x_test_svd_NER_undersample)
accuracy_hmm_stem_undersample = accuracy_score(y_test_undersampled, y_pred_stem_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_undersample}")

testing HMM BoW Accuracy: 0.04792140888942135


In [125]:
results_NER_undersample['undersample HMM']=0.04792140888942135

## with tf-idf

In [126]:
for model_name, model in models_NER_undersample.items():
    model.fit(x_train_NER_undersample_scaled_tfidf, y_train_undersampled)

    y_pred_train = model.predict(x_train_NER_undersample_scaled_tfidf)
    accuracy_train=accuracy_score(y_train_undersampled, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_NER_undersample_scaled_tfidf)
    accuracy=accuracy_score(y_test_undersampled, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_undersampled, y_pred))
    print('-'*50)
    results_NER_undersample['(tf-idf) undersample '+model_name] = accuracy

Results for SVM Training: accuracy=0.2318866624734178
Results for SVM Testing: accuracy=0.23487480531927638
              precision    recall  f1-score   support

           1       0.29      0.36      0.32      2000
           2       0.19      0.81      0.31      2000
           3       0.28      0.41      0.33      2000
           4       0.28      0.23      0.25      1980
           5       0.25      0.14      0.18      1963
           6       0.00      0.00      0.00      1269
           7       0.00      0.00      0.00      1268
           8       0.25      0.00      0.00      1198
           9       0.23      0.01      0.01      1016
          10       0.21      0.00      0.01      2000

    accuracy                           0.23     16694
   macro avg       0.20      0.20      0.14     16694
weighted avg       0.21      0.23      0.17     16694

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.15670770060203074
Results for Multi

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Results for MLP Training: accuracy=0.23061371192380267
Results for MLP Testing: accuracy=0.23391637714148797
              precision    recall  f1-score   support

           1       0.28      0.35      0.31      2000
           2       0.19      0.83      0.31      2000
           3       0.28      0.42      0.34      2000
           4       0.31      0.21      0.25      1980
           5       0.24      0.14      0.18      1963
           6       0.00      0.00      0.00      1269
           7       0.00      0.00      0.00      1268
           8       0.40      0.00      0.00      1198
           9       0.08      0.00      0.00      1016
          10       0.22      0.00      0.01      2000

    accuracy                           0.23     16694
   macro avg       0.20      0.20      0.14     16694
weighted avg       0.22      0.23      0.17     16694

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### HMM

In [127]:
hmm_NER_undersample_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

#### dimension reduction (66774,18)

In [129]:
svd_NER = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_NER_undersample_tfidf = svd_NER.fit_transform(x_train_NER_undersample_scaled_tfidf)
x_test_svd_NER_undersample_tfidf = svd_NER.transform(x_test_NER_undersample_scaled_tfidf)

scaler = StandardScaler()
x_train_svd_NER_undersample_tfidf = scaler.fit_transform(x_train_svd_NER_undersample_tfidf)
x_test_svd_NER_undersample_tfidf = scaler.transform(x_test_svd_NER_undersample_tfidf)

#### apply HMM

In [130]:
lengths_train = [1]*x_train_svd_NER_undersample_tfidf.shape[0]
lengths_test = [1]*x_test_svd_NER_undersample_tfidf.shape[0]

In [131]:
hmm_NER_undersample_tfidf.fit(x_train_svd_NER_undersample_tfidf)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [132]:
y_pred_stem_tfidf_undersample=hmm_NER_undersample_tfidf.predict(x_train_svd_NER_undersample_tfidf)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_train_undersampled, y_pred_stem_tfidf_undersample)   
print(f"training HMM BoW Accuracy : {accuracy_hmm_stem_tfidf_undersample}")

training HMM BoW Accuracy : 0.0818881600622997


In [133]:
y_pred_stem_tfidf_undersample=hmm_NER_undersample_tfidf.predict(x_test_svd_NER_undersample_tfidf,lengths=lengths_test)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy with lengths: 0.08482089373427579


In [134]:
y_pred_stem_tfidf_undersample=hmm_NER_undersample_tfidf.predict(x_test_svd_NER_undersample_tfidf)
accuracy_hmm_stem_tfidf_undersample = accuracy_score(y_test_undersampled, y_pred_stem_tfidf_undersample)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_stem_tfidf_undersample}")

testing HMM BoW Accuracy: 0.08056786869533965


In [136]:
results_NER_undersample['(tf-idf)undersample HMM']=0.08482089373427579

### save the results

In [138]:
joblib.dump(results_NER_undersample,'results_NER_undersample')
results_NER_undersample

{'undersample SVM': 0.23415598418593506,
 'undersample MultinomialNB': 0.1590391757517671,
 'undersample MLP': 0.23439559123038217,
 'undersample HMM': 0.04792140888942135,
 '(tf-idf) undersample SVM': 0.23487480531927638,
 '(tf-idf) undersample MultinomialNB': 0.15832035461842578,
 '(tf-idf) undersample MLP': 0.23391637714148797,
 '(tf-idf)undersample HMM': 0.08482089373427579}

In [139]:
with open("results_NER_undersample.txt", "w", encoding="utf-8") as f:
    for key, value in results_NER_undersample.items():
        f.write(f"{key}: {value}\n")

# ______________________________________________________________________________________

# resampled Data (rosrus)

In [140]:
results_NER_rosrus={}

## NER feature Extraction + scaling

### with countvictorizer

In [141]:
vectorizer_NER_rosrus = CountVectorizer(tokenizer=lambda x:NER_features(x))
x_train_NER_rosrus = vectorizer_NER_rosrus.fit_transform(x_train_bal)
x_test_NER_rosrus= vectorizer_NER_rosrus.transform(x_test_bal)

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [142]:
scaler_rosrus = StandardScaler(with_mean=False)
x_train_NER_rosrus_scaled = scaler_rosrus.fit_transform(x_train_NER_rosrus)
x_test_NER_rosrus_scaled  = scaler_rosrus.transform(x_test_NER_rosrus)

In [143]:
x_train_NER_rosrus_scaled.shape,x_test_NER_rosrus_scaled.shape

((130000, 18), (41905, 18))

### with tf-idf

In [145]:
vectorizer_NER_rosrus_tfidf = TfidfVectorizer(tokenizer=lambda x:NER_features(x))
x_train_NER_rosrus_tfidf = vectorizer_NER_rosrus_tfidf.fit_transform(x_train_bal)
x_test_NER_rosrus_tfidf= vectorizer_NER_rosrus_tfidf.transform(x_test_bal)

In [146]:
scaler_rosrus_tfidf = StandardScaler(with_mean=False)
x_train_NER_rosrus_tfidf_scaled = scaler_rosrus_tfidf.fit_transform(x_train_NER_rosrus_tfidf)
x_test_NER_rosrus_tfidf_scaled  = scaler_rosrus_tfidf.transform(x_test_NER_rosrus_tfidf)

In [147]:
x_train_NER_rosrus_tfidf_scaled.shape,x_test_NER_rosrus_tfidf_scaled.shape

((130000, 18), (41905, 18))

## Models

In [148]:
models_NER_rosrus={
    'SVM': SVC(),
    'MultinomialNB': MultinomialNB(),
    'MLP': MLPClassifier(hidden_layer_sizes=(32,16),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42),
}

### with countvectorizer

In [149]:
for model_name, model in models_NER_rosrus.items():
    model.fit(x_train_NER_rosrus_scaled, y_train_bal)

    y_pred_train = model.predict(x_train_NER_rosrus_scaled)
    accuracy_train=accuracy_score(y_train_bal, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_NER_rosrus_scaled)
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_NER_rosrus['rorus '+model_name] = accuracy

Results for SVM Training: accuracy=0.1978076923076923
Results for SVM Testing: accuracy=0.1775921727717456
              precision    recall  f1-score   support

           1       0.37      0.34      0.35      7120
           2       0.14      0.80      0.23      3589
           3       0.21      0.38      0.27      3473
           4       0.12      0.20      0.15      1980
           5       0.10      0.14      0.12      1963
           6       0.05      0.00      0.01      1269
           7       0.04      0.00      0.00      1268
           8       0.04      0.02      0.03      1198
           9       0.07      0.06      0.07      1016
          10       0.55      0.00      0.01     19029

    accuracy                           0.18     41905
   macro avg       0.17      0.20      0.12     41905
weighted avg       0.36      0.18      0.12     41905

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.1311153846153846
Results for Multino

#### HMM

In [150]:
hmm_NER_rosrus=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

##### dimension reduction (,35)

In [151]:
svd_NER = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_NER_rosrus = svd_NER.fit_transform(x_train_NER_rosrus_scaled)
x_test_svd_NER_rosrus = svd_NER.transform(x_test_NER_rosrus_scaled)

scaler = StandardScaler()
x_train_svd_NER_rosrus = scaler.fit_transform(x_train_svd_NER_rosrus)
x_test_svd_NER_rosrus = scaler.transform(x_test_svd_NER_rosrus)

In [152]:
lengths_train = [1]*x_train_svd_NER_rosrus.shape[0]
lengths_test = [1]*x_test_svd_NER_rosrus.shape[0]

In [153]:
hmm_NER_rosrus.fit(x_train_svd_NER_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [154]:
y_pred_NER_rosrus=hmm_NER_rosrus.predict(x_train_svd_NER_rosrus,lengths=lengths_train)
accuracy_hmm_NER_rosrus = accuracy_score(y_train_bal, y_pred_NER_rosrus)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_NER_rosrus}")

training HMM BoW Accuracy with lengths: 0.05173846153846154


In [155]:
y_pred_NER_rosrus=hmm_NER_rosrus.predict(x_test_svd_NER_rosrus,lengths=lengths_test)
accuracy_hmm_NER_rosrus = accuracy_score(y_test_bal, y_pred_NER_rosrus)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_NER_rosrus}")

testing HMM BoW Accuracy with lengths: 0.08769836535019687


In [156]:
y_pred_NER_rosrus=hmm_NER_rosrus.predict(x_test_svd_NER_rosrus)
accuracy_hmm_NER_rosrus = accuracy_score(y_test_bal, y_pred_NER_rosrus)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_NER_rosrus}")

testing HMM BoW Accuracy: 0.091301753967307


In [166]:
results_NER_rosrus['rosrus HMM StopWords']=0.091301753967307

### with tf-idf

In [157]:
for model_name, model in models_NER_rosrus.items():
    model.fit(x_train_NER_rosrus_tfidf_scaled, y_train_bal)

    y_pred_train = model.predict(x_train_NER_rosrus_tfidf_scaled)
    accuracy_train=accuracy_score(y_train_bal, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_NER_rosrus_tfidf_scaled)
    accuracy=accuracy_score(y_test_bal, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test_bal, y_pred))
    print('-'*50)
    results_NER_rosrus['(tf-idf) rorus '+model_name] = accuracy

Results for SVM Training: accuracy=0.19603846153846155
Results for SVM Testing: accuracy=0.17408423815773774
              precision    recall  f1-score   support

           1       0.37      0.31      0.34      7120
           2       0.14      0.80      0.23      3589
           3       0.20      0.40      0.27      3473
           4       0.12      0.22      0.16      1980
           5       0.10      0.14      0.12      1963
           6       0.04      0.00      0.00      1269
           7       0.04      0.00      0.00      1268
           8       0.04      0.02      0.03      1198
           9       0.04      0.04      0.04      1016
          10       0.62      0.00      0.01     19029

    accuracy                           0.17     41905
   macro avg       0.17      0.19      0.12     41905
weighted avg       0.39      0.17      0.12     41905

--------------------------------------------------
Results for MultinomialNB Training: accuracy=0.1301
Results for MultinomialNB Tes

#### HMM

In [158]:
hmm_NER_rosrus_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

##### dimension reduction (,35)

In [159]:
svd_NER_tfidf = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_NER_tfidf_rosrus = svd_NER_tfidf.fit_transform(x_train_NER_rosrus_tfidf_scaled)
x_test_svd_NER_tfidf_rosrus = svd_NER_tfidf.transform(x_test_NER_rosrus_tfidf_scaled)

scaler = StandardScaler()
x_train_svd_NER_tfidf_rosrus = scaler.fit_transform(x_train_svd_NER_tfidf_rosrus)
x_test_svd_NER_tfidf_rosrus = scaler.transform(x_test_svd_NER_tfidf_rosrus)

In [160]:
lengths_train = [1]*x_train_svd_NER_tfidf_rosrus.shape[0]
lengths_test = [1]*x_test_svd_NER_tfidf_rosrus.shape[0]

In [161]:
hmm_NER_rosrus_tfidf.fit(x_train_svd_NER_tfidf_rosrus)

GaussianHMM(n_components=10, n_iter=100, random_state=42)

In [162]:
y_pred_NER_rosrus_tfidf=hmm_NER_rosrus_tfidf.predict(x_train_svd_NER_tfidf_rosrus,lengths=lengths_train)
accuracy_hmm_NER_rosrus_tfidf = accuracy_score(y_train_bal, y_pred_NER_rosrus_tfidf)   
print(f"training HMM BoW Accuracy with lengths: {accuracy_hmm_NER_rosrus_tfidf}")

training HMM BoW Accuracy with lengths: 0.0


In [163]:
y_pred_NER_rosrus_tfidf=hmm_NER_rosrus_tfidf.predict(x_test_svd_NER_tfidf_rosrus,lengths=lengths_test)
accuracy_hmm_NER_rosrus_tfidf = accuracy_score(y_test_bal, y_pred_NER_rosrus_tfidf)   
print(f"testing HMM BoW Accuracy with lengths: {accuracy_hmm_NER_rosrus_tfidf}")

testing HMM BoW Accuracy with lengths: 0.0


In [164]:
y_pred_NER_rosrus_tfidf=hmm_NER_rosrus_tfidf.predict(x_test_svd_NER_tfidf_rosrus,)
accuracy_hmm_NER_rosrus_tfidf = accuracy_score(y_test_bal, y_pred_NER_rosrus_tfidf)   
print(f"testing HMM BoW Accuracy: {accuracy_hmm_NER_rosrus_tfidf}")

testing HMM BoW Accuracy: 0.028755518434554348


In [167]:
results_NER_rosrus['(tf-idf) rosrus HMM']=0.028755518434554348

In [169]:
results_NER_rosrus

{'rorus SVM': 0.1775921727717456,
 'rorus MultinomialNB': 0.12969812671518913,
 'rorus MLP': 0.1727240186135306,
 '(tf-idf) rorus SVM': 0.17408423815773774,
 '(tf-idf) rorus MultinomialNB': 0.12936403770433122,
 '(tf-idf) rorus MLP': 0.17162629757785466,
 'rosrus HMM StopWords': 0.091301753967307,
 '(tf-idf) rosrus HMM': 0.028755518434554348}

## Save results

In [170]:
joblib.dump(results_NER_rosrus,'results_NER_rosrus')

['results_NER_rosrus']

In [171]:
results_NER_rosrus=joblib.load('results_NER_rosrus')
results_NER_rosrus

{'rorus SVM': 0.1775921727717456,
 'rorus MultinomialNB': 0.12969812671518913,
 'rorus MLP': 0.1727240186135306,
 '(tf-idf) rorus SVM': 0.17408423815773774,
 '(tf-idf) rorus MultinomialNB': 0.12936403770433122,
 '(tf-idf) rorus MLP': 0.17162629757785466,
 'rosrus HMM StopWords': 0.091301753967307,
 '(tf-idf) rosrus HMM': 0.028755518434554348}

In [172]:
with open("results_NER_rosrus.txt", "w", encoding="utf-8") as f:
    for key, value in results_NER_rosrus.items():
        f.write(f"{key}: {value}\n")